# Scale-Robust PIELM with Sinusoidal Basis

**Goal:** Demonstrate that log-uniform (power-law, alpha~1) weight initialization + sin() activation produces a scale-robust PIELM that doesn't need expensive Rm tuning.

**Key experiments:**
1. Full PIELM benchmark (power-law vs Gaussian vs Uniform) — fair comparison, no u_true in training
2. Scale sweep — controlling for Dong (2022) critique: at each distribution's optimal scale, compare
3. Alpha ablation — find optimal power-law exponent, show alpha=2.0 is suboptimal
4. Condition number analysis — theoretical backbone
5. Frequency spectrum — visualize why log-uniform gives multi-scale coverage

All benchmarks use **PIELM** (physics-informed). u_true is only for evaluation, never training.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import math
from typing import Tuple, Callable, Dict, Any, List

torch.set_default_dtype(torch.float64)

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 120,
    'figure.facecolor': 'white',
})

COLORS = {
    'power': '#E63946',
    'normal': '#457B9D',
    'uniform': '#2A9D8F',
}
LABELS = {
    'power': 'Log-Uniform (Power-Law)',
    'normal': 'Gaussian',
    'uniform': 'Uniform',
}

print('Setup complete.')

## Core PIELM Building Blocks

Sinusoidal activation `sin(wx+b)` gives **exact** analytical derivatives:
- `phi'(x) = w * cos(wx+b)`
- `phi''(x) = -w^2 * sin(wx+b)`

No finite difference, no autodiff. The PDE operator applied to the basis stays linear in beta.

In [ ]:
# ── Weight Initialization ──

def init_weights(input_dim, hidden_dim, init_type='power', scale=30.0,
                 alpha=1.0, dtype=torch.float64):
    """Generate frozen random weights.
    
    init_type:
        'power'   : log-uniform on [1, scale] with random sign  (alpha~1 power-law)
        'normal'  : N(0, scale)
        'uniform' : U(-scale, scale)
        'power_alpha': true P(w)~|w|^{-alpha} on [w_min, scale] via inverse CDF
    """
    if init_type == 'uniform':
        W = (2*torch.rand(input_dim, hidden_dim, dtype=dtype) - 1) * scale
        b = (2*torch.rand(1, hidden_dim, dtype=dtype) - 1) * scale
    elif init_type == 'normal':
        W = torch.randn(input_dim, hidden_dim, dtype=dtype) * scale
        b = torch.randn(1, hidden_dim, dtype=dtype) * scale
    elif init_type == 'power':
        log_w = torch.rand(input_dim, hidden_dim, dtype=dtype) * torch.log10(
            torch.tensor(scale, dtype=dtype))
        sign = 2*torch.randint(0, 2, (input_dim, hidden_dim)).to(dtype) - 1
        W = 10.0**log_w * sign
        b = (2*torch.rand(1, hidden_dim, dtype=dtype) - 1) * torch.pi
    elif init_type == 'power_alpha':
        w_min = 0.1
        u = torch.rand(input_dim, hidden_dim, dtype=dtype)
        if abs(alpha - 1.0) < 1e-6:
            W = w_min * (scale / w_min)**u
        else:
            W = (w_min**(1-alpha) + u*(scale**(1-alpha) - w_min**(1-alpha)))**(1.0/(1-alpha))
        sign = 2*torch.randint(0, 2, (input_dim, hidden_dim)).to(dtype) - 1
        W = W * sign
        b = (2*torch.rand(1, hidden_dim, dtype=dtype) - 1) * torch.pi
    else:
        raise ValueError(f'Unknown init_type: {init_type}')
    return W, b


# ── Feature Matrices ──

def H_mat(x, W, b):
    """H[i,j] = sin(w_j * x_i + b_j)"""
    return torch.sin(x @ W + b)

def H_d1(x, W, b):
    """First derivative: w_j * cos(w_j * x_i + b_j)"""
    return W * torch.cos(x @ W + b)

def H_d2(x, W, b):
    """Second derivative: -w_j^2 * sin(w_j * x_i + b_j)"""
    w_sq = (W**2).sum(dim=0, keepdim=True)
    return -w_sq * H_mat(x, W, b)


# ── PDE System Builders ──

def build_poisson(x_int, x_bc, f_int, u_bc, W, b, bc_w=100.0):
    """-u'' = f, Dirichlet BCs. Returns (A, rhs)."""
    H_pde = -H_d2(x_int, W, b)          # w^2 * sin(...)
    H_bc  = H_mat(x_bc, W, b)
    A   = torch.cat([H_pde, bc_w * H_bc], dim=0)
    rhs = torch.cat([f_int, bc_w * u_bc], dim=0)
    return A, rhs

def build_helmholtz(x_int, x_bc, f_int, u_bc, W, b, k, bc_w=100.0):
    """-u'' - k^2*u = f, Dirichlet BCs."""
    H_base = H_mat(x_int, W, b)
    w_sq = (W**2).sum(dim=0, keepdim=True)
    H_pde = (w_sq - k**2) * H_base
    H_bc  = H_mat(x_bc, W, b)
    A   = torch.cat([H_pde, bc_w * H_bc], dim=0)
    rhs = torch.cat([f_int, bc_w * u_bc], dim=0)
    return A, rhs

def build_advdiff(x_int, x_bc, f_int, u_bc, W, b, eps, bc_w=100.0):
    """-eps*u'' + u' = f, Dirichlet BCs."""
    H_pde = -eps * H_d2(x_int, W, b) + H_d1(x_int, W, b)
    H_bc  = H_mat(x_bc, W, b)
    A   = torch.cat([H_pde, bc_w * H_bc], dim=0)
    rhs = torch.cat([f_int, bc_w * u_bc], dim=0)
    return A, rhs


# ── Solver ──

def solve(A, rhs, lambd=1e-10):
    """Regularised least-squares: (A'A + lam*I)^{-1} A'rhs"""
    ATA = A.t() @ A + lambd * torch.eye(A.shape[1], dtype=A.dtype)
    return torch.linalg.solve(ATA, A.t() @ rhs)

def predict(x, W, b, beta):
    return H_mat(x, W, b) @ beta

print('PIELM building blocks ready.')

In [ ]:
# ── Benchmark Problem Definitions ──

def make_poisson_simple():
    """u = sin(pi*x), -u'' = pi^2 sin(pi*x)"""
    f = lambda x: math.pi**2 * torch.sin(math.pi * x)
    u = lambda x: torch.sin(math.pi * x)
    return dict(name='Poisson (sin pi x)', pde='poisson', f_fn=f, u_fn=u,
                domain=(-1,1), bc=(0.0, 0.0))

def make_poisson_multifreq():
    """u = sin(pi*x) + 0.3*sin(3*pi*x) + 0.1*sin(7*pi*x)"""
    def f(x):
        return (math.pi**2 * torch.sin(math.pi*x)
                + 0.3*(3*math.pi)**2 * torch.sin(3*math.pi*x)
                + 0.1*(7*math.pi)**2 * torch.sin(7*math.pi*x))
    def u(x):
        return (torch.sin(math.pi*x) + 0.3*torch.sin(3*math.pi*x)
                + 0.1*torch.sin(7*math.pi*x))
    return dict(name='Multi-Freq Poisson', pde='poisson', f_fn=f, u_fn=u,
                domain=(-1,1), bc=(0.0, 0.0))

def make_helmholtz():
    """u = sin(pi*x)*exp(-4x^2), -u''-k^2*u = f, k=10"""
    k = 10.0
    def u(x):
        return torch.sin(math.pi*x) * torch.exp(-4*x**2)
    def f(x):
        s = torch.sin(math.pi*x)
        c = torch.cos(math.pi*x)
        e = torch.exp(-4*x**2)
        u_pp = (-math.pi**2 * s * e
                + 2 * math.pi * c * (-8*x) * e
                + s * (-8 + 64*x**2) * e)
        return -u_pp - k**2 * s * e
    return dict(name=f'Helmholtz (k={k})', pde='helmholtz', f_fn=f, u_fn=u,
                domain=(-1,1), bc=(0.0, 0.0), k=k)

def make_poisson_highfreq():
    """u = sin(10*pi*x)"""
    w = 10*math.pi
    f = lambda x: w**2 * torch.sin(w * x)
    u = lambda x: torch.sin(w * x)
    return dict(name='High-Freq Poisson (10pi)', pde='poisson', f_fn=f, u_fn=u,
                domain=(-1,1), bc=(0.0, 0.0))

def make_advdiff():
    """Boundary layer: -eps*u'' + u' = 0, u(0)=0, u(1)=1, eps=0.02"""
    eps = 0.02
    def u(x):
        return (torch.exp(x/eps) - 1) / (math.exp(1/eps) - 1)
    f = lambda x: torch.zeros_like(x)
    return dict(name='Advection-Diffusion (eps=0.02)', pde='advdiff', f_fn=f, u_fn=u,
                domain=(0.0, 1.0), bc=(0.0, 1.0), eps=eps)

BENCHMARKS = [make_poisson_simple, make_poisson_multifreq, make_helmholtz,
              make_poisson_highfreq, make_advdiff]

print(f'{len(BENCHMARKS)} benchmarks defined.')

In [ ]:
# ── Helper: run a single PIELM trial ──

def run_trial(bench, init_type, scale, hidden_dim, n_interior, seed,
              alpha=1.0, lambd=1e-10, bc_w=100.0):
    """Run one PIELM trial. Returns dict with error, cond, time."""
    dtype = torch.float64
    a, b_dom = bench['domain']
    
    x_int = torch.linspace(a, b_dom, n_interior+2, dtype=dtype)[1:-1].unsqueeze(1)
    x_bc  = torch.tensor([[a], [b_dom]], dtype=dtype)
    f_int = bench['f_fn'](x_int)
    u_bc  = torch.tensor([[bench['bc'][0]], [bench['bc'][1]]], dtype=dtype)
    
    torch.manual_seed(seed)
    t0 = time.perf_counter()
    
    W, bias = init_weights(1, hidden_dim, init_type, scale, alpha=alpha, dtype=dtype)
    
    if bench['pde'] == 'poisson':
        A, rhs = build_poisson(x_int, x_bc, f_int, u_bc, W, bias, bc_w)
    elif bench['pde'] == 'helmholtz':
        A, rhs = build_helmholtz(x_int, x_bc, f_int, u_bc, W, bias, bench['k'], bc_w)
    elif bench['pde'] == 'advdiff':
        A, rhs = build_advdiff(x_int, x_bc, f_int, u_bc, W, bias, bench['eps'], bc_w)
    
    beta = solve(A, rhs, lambd)
    elapsed = time.perf_counter() - t0
    
    # Evaluate
    x_test = torch.linspace(a, b_dom, 1000, dtype=dtype).unsqueeze(1)
    u_pred = predict(x_test, W, bias, beta)
    u_true = bench['u_fn'](x_test)
    err = (torch.norm(u_pred - u_true) / torch.norm(u_true)).item()
    
    try:
        cond = torch.linalg.cond(A).item()
    except:
        cond = float('inf')
    
    return dict(error=err, cond=cond, time=elapsed,
                u_pred=u_pred.detach(), u_true=u_true.detach(), x_test=x_test.detach())


def run_multi_seed(bench, init_type, scale, hidden_dim=400, n_interior=200,
                   n_seeds=20, alpha=1.0, **kwargs):
    """Run n_seeds trials, return aggregated stats."""
    results = [run_trial(bench, init_type, scale, hidden_dim, n_interior, s,
                         alpha=alpha, **kwargs) for s in range(n_seeds)]
    errors = [r['error'] for r in results]
    conds  = [r['cond']  for r in results]
    times  = [r['time']  for r in results]
    return dict(
        error_mean=np.mean(errors), error_std=np.std(errors),
        error_median=np.median(errors), errors=errors,
        cond_mean=np.mean(conds), cond_std=np.std(conds), conds=conds,
        time_mean=np.mean(times), times=times,
        last_pred=results[0]['u_pred'], last_true=results[0]['u_true'],
        x_test=results[0]['x_test'],
    )

print('Trial runner ready.')

---
## Experiment 7: Multi-Scale Edge Characterisation

**Goal:** Identify *which problems* give log-uniform a clear edge at tight budgets (h <= 30),
and characterise the relationship between **spectral gap** (ratio of highest to lowest
frequency in the solution) and the **critical h*** below which log-uniform dominates.

**Hypothesis:** The edge appears when the solution spans >= 1 frequency decade.
Gaussian/Uniform waste neurons by clustering in one band, leaving other decades empty.

We test a ladder of Poisson problems with increasing spectral gap,
from single-frequency (no gap) to 2-decade multi-scale solutions.

In [ ]:
# ── Experiment 7: Multi-Scale Benchmark Ladder ──
#
# Each problem is  -u'' = f  with manufactured solution u(x).
# We vary the spectral content to control the frequency gap.

def make_single_freq(omega=math.pi):
    """Single frequency: u = sin(omega*x). Gap = 0 decades."""
    f = lambda x: omega**2 * torch.sin(omega * x)
    u = lambda x: torch.sin(omega * x)
    return dict(name=f'Single (w={omega/math.pi:.0f}pi)', pde='poisson', f_fn=f, u_fn=u,
                domain=(-1,1), bc=(0.0, 0.0), gap_decades=0.0, freqs=[omega])

def make_two_freq(w1, w2, a2=0.3):
    """Two frequencies: u = sin(w1*x) + a2*sin(w2*x)."""
    gap = math.log10(max(w1,w2)/min(w1,w2))
    def f(x):
        return w1**2*torch.sin(w1*x) + a2*w2**2*torch.sin(w2*x)
    def u(x):
        return torch.sin(w1*x) + a2*torch.sin(w2*x)
    return dict(name=f'2-freq ({w1/math.pi:.0f}pi,{w2/math.pi:.0f}pi)', pde='poisson',
                f_fn=f, u_fn=u, domain=(-1,1), bc=(0.0, 0.0),
                gap_decades=gap, freqs=[w1, w2])

def make_three_freq(w1, w2, w3, a2=0.3, a3=0.1):
    """Three frequencies: u = sin(w1*x) + a2*sin(w2*x) + a3*sin(w3*x)."""
    gap = math.log10(max(w1,w2,w3)/min(w1,w2,w3))
    def f(x):
        return (w1**2*torch.sin(w1*x) + a2*w2**2*torch.sin(w2*x)
                + a3*w3**2*torch.sin(w3*x))
    def u(x):
        return torch.sin(w1*x) + a2*torch.sin(w2*x) + a3*torch.sin(w3*x)
    return dict(name=f'3-freq ({w1/math.pi:.0f},{w2/math.pi:.0f},{w3/math.pi:.0f})pi',
                pde='poisson', f_fn=f, u_fn=u, domain=(-1,1), bc=(0.0, 0.0),
                gap_decades=gap, freqs=[w1, w2, w3])

# ── Benchmark ladder: increasing spectral gap ──
p = math.pi
EXP7_BENCHMARKS = [
    # 0 decades (single frequency)
    make_single_freq(p),
    # ~0.5 decades  (pi and 3*pi)
    make_two_freq(p, 3*p),
    # ~0.85 decades (pi and 7*pi)  — the paper's multi-freq Poisson
    make_three_freq(p, 3*p, 7*p),
    # ~1.0 decade   (pi and 10*pi)
    make_two_freq(p, 10*p),
    # ~1.3 decades  (pi, 5*pi, 20*pi)
    make_three_freq(p, 5*p, 20*p),
    # ~1.5 decades  (pi, 10*pi, 30*pi)
    make_three_freq(p, 10*p, 30*p),
    # ~1.7 decades  (pi and 50*pi)
    make_two_freq(p, 50*p, a2=0.1),
    # ~2.0 decades  (pi and 100*pi)
    make_two_freq(p, 100*p, a2=0.1),
]

print(f'Spectral ladder: {len(EXP7_BENCHMARKS)} problems')
for b in EXP7_BENCHMARKS:
    print(f"  {b['name']:<35s} gap = {b['gap_decades']:.2f} decades  "
          f"freqs = {[f'{w/math.pi:.0f}pi' for w in b['freqs']]}")

# ── Run experiment: sweep h for each problem ──
EXP7_H = [10, 15, 20, 25, 30, 40, 60, 80, 120, 200, 400]
EXP7_SEEDS = 20

exp7 = {}
for bench in EXP7_BENCHMARKS:
    name = bench['name']
    max_freq = max(bench['freqs'])
    # Scale: cover highest frequency with margin
    sc = max(30.0, max_freq * 1.2)
    # Collocation points: need >= 2*max_freq/pi per unit length for Nyquist
    n_int = max(200, int(max_freq / math.pi * 4))
    print(f'\n--- {name} (scale={sc:.0f}, N_int={n_int}) ---')
    exp7[name] = {'bench': bench, 'scale': sc, 'n_int': n_int}
    for init in INIT_TYPES:
        hdim_data = []
        for h in EXP7_H:
            r = run_multi_seed(bench, init, sc, h, n_int, EXP7_SEEDS)
            hdim_data.append(dict(h=h, **r))
        exp7[name][init] = hdim_data
        r20 = next(d for d in hdim_data if d['h'] == 20)
        r30 = next(d for d in hdim_data if d['h'] == 30)
        print(f"  {init:<10s}  h=20: {r20['error_median']:.2e}  "
              f"h=30: {r30['error_median']:.2e}  "
              f"h=400: {hdim_data[-1]['error_median']:.2e}")

print('\nExperiment 7 complete.')

In [ ]:
# ── Experiment 7 Plots ──
#
# (A) Error-vs-h curves for each spectral gap level
# (B) "Edge map": power-law advantage ratio vs spectral gap
#
# Filter rule: skip any h where ALL methods have error > 0.5
# (nobody can solve it — noise, not signal)

ERR_FLOOR = 0.5  # if all inits above this, the problem is unsolvable at that h

n_bench = len(exp7)
n_cols = min(4, n_bench)
n_rows = (n_bench + n_cols - 1) // n_cols

# (A) Error vs h curves
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4.5*n_rows))
axes = np.array(axes).flatten()

for idx, (name, data) in enumerate(exp7.items()):
    ax = axes[idx]
    gap = data['bench']['gap_decades']
    # Find valid h values (at least one init below ERR_FLOOR)
    valid_h = set()
    for init in INIT_TYPES:
        for d in data[init]:
            if d['error_median'] < ERR_FLOOR:
                valid_h.add(d['h'])
    for init in INIT_TYPES:
        hd = data[init]
        hs = [d['h'] for d in hd if d['h'] in valid_h]
        meds = [d['error_median'] for d in hd if d['h'] in valid_h]
        stds = [d['error_std'] for d in hd if d['h'] in valid_h]
        if not hs:
            continue
        ax.semilogy(hs, meds, 'o-', color=COLORS[init], label=LABELS[init],
                    markersize=4, linewidth=1.5)
        ax.fill_between(hs,
                        [max(m-s, 1e-15) for m,s in zip(meds,stds)],
                        [m+s for m,s in zip(meds,stds)],
                        color=COLORS[init], alpha=0.12)
    ax.axvline(30, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlabel('Hidden Neurons (h)')
    ax.set_ylabel('L2 Relative Error')
    ax.set_title(f'{name}\n(gap={gap:.2f} dec)', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for idx in range(len(exp7), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Exp 7: Error vs Neuron Budget — Increasing Spectral Gap\n'
             '(dashed line = h=30; points where all methods fail are omitted)',
             fontsize=13)
fig.tight_layout()
plt.savefig('exp7_spectral_ladder.png', dpi=150, bbox_inches='tight')
plt.show()

# ──────────────────────────────────────────────
# (B) Advantage ratio vs spectral gap at h=20 and h=30
#     Only include points where power-law error < ERR_FLOOR
# ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, h_target in zip(axes, [20, 30]):
    gaps, ratio_g, ratio_u = [], [], []
    labels_plot = []
    for name, data in exp7.items():
        gap = data['bench']['gap_decades']
        h_idx = next((i for i, d in enumerate(data['power']) if d['h'] == h_target), None)
        if h_idx is None:
            continue
        e_p = data['power'][h_idx]['error_median']
        e_g = data['normal'][h_idx]['error_median']
        e_u = data['uniform'][h_idx]['error_median']
        # Skip if power-law itself fails
        if e_p >= ERR_FLOOR:
            continue
        gaps.append(gap)
        ratio_g.append(e_g / max(e_p, 1e-15))
        ratio_u.append(e_u / max(e_p, 1e-15))
        labels_plot.append(name)

    if not gaps:
        ax.text(0.5, 0.5, f'No valid data at h={h_target}',
                transform=ax.transAxes, ha='center')
        continue

    ax.semilogy(gaps, ratio_g, 's-', color=COLORS['normal'],
                label='Gaussian / Log-Uniform', markersize=7, linewidth=2)
    ax.semilogy(gaps, ratio_u, '^-', color=COLORS['uniform'],
                label='Uniform / Log-Uniform', markersize=7, linewidth=2)
    ax.axhline(1, color='gray', ls=':', lw=1)
    ax.set_xlabel('Spectral Gap (decades)')
    ax.set_ylabel('Error Ratio (higher = log-uniform wins more)')
    ax.set_title(f'Log-Uniform Advantage at h = {h_target}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    for g, r, lbl in zip(gaps, ratio_g, labels_plot):
        if r > 5:
            ax.annotate(f'{r:.0f}x', (g, r), textcoords='offset points',
                        xytext=(5, 5), fontsize=8, color=COLORS['normal'])

fig.suptitle('Exp 7: Log-Uniform Advantage vs Spectral Gap\n'
             '(only shown where power-law error < 0.5)', fontsize=13)
fig.tight_layout()
plt.savefig('exp7_advantage_vs_gap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Experiment 7c: Critical h* — smallest h where error < threshold ──
#
# For each benchmark and init type, find the smallest h where:
#   (a) the method's own error < threshold, AND
#   (b) the method's error < 0.5 (i.e. it's actually solving the problem)

THRESHOLDS = [1e-2, 1e-4, 1e-8]

fig, axes = plt.subplots(1, len(THRESHOLDS), figsize=(6*len(THRESHOLDS), 5))

for ax, thresh in zip(axes, THRESHOLDS):
    for init in INIT_TYPES:
        gaps_plot, hstars = [], []
        for name, data in exp7.items():
            gap = data['bench']['gap_decades']
            hd = data[init]
            h_star = None
            for d in hd:
                if d['error_median'] < thresh:
                    h_star = d['h']
                    break
            if h_star is not None:
                gaps_plot.append(gap)
                hstars.append(h_star)
        if gaps_plot:
            ax.plot(gaps_plot, hstars, 'o-', color=COLORS[init],
                    label=LABELS[init], markersize=6, linewidth=2)

    ax.set_xlabel('Spectral Gap (decades)')
    ax.set_ylabel('Minimum h needed')
    ax.set_title(f'h* for L2 error < {thresh:.0e}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Exp 7: Neuron Efficiency — h* vs Spectral Gap\n'
             '(lower = fewer neurons needed)', fontsize=13)
fig.tight_layout()
plt.savefig('exp7_critical_h.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print summary table ──
print('\n' + '=' * 110)
print('CRITICAL h* SUMMARY (minimum h for median error < threshold)')
print('=' * 110)
for thresh in THRESHOLDS:
    print(f'\nThreshold: {thresh:.0e}')
    print(f'{"Problem":<35s} {"Gap":<6s} {"Power-Law":<12s} {"Gaussian":<12s} '
          f'{"Uniform":<12s} {"PL saves vs G":<15s}')
    print('-' * 95)
    for name, data in exp7.items():
        gap = data['bench']['gap_decades']
        hstars = {}
        for init in INIT_TYPES:
            hd = data[init]
            h_star = '>400'
            for d in hd:
                if d['error_median'] < thresh:
                    h_star = d['h']
                    break
            hstars[init] = h_star
        hp = hstars['power']
        hg = hstars['normal']
        if isinstance(hp, int) and isinstance(hg, int) and hp > 0:
            savings = f'{hg/hp:.1f}x fewer'
        elif isinstance(hp, int) and hg == '>400':
            savings = f'>{400/hp:.0f}x fewer'
        else:
            savings = '—'
        print(f'{name:<35s} {gap:<6.2f} {str(hstars["power"]):<12s} '
              f'{str(hstars["normal"]):<12s} {str(hstars["uniform"]):<12s} '
              f'{savings:<15s}')